# Ch04 RAG & Embeddings — GPU Supplement

This notebook compares CPU-viable vs GPU-beneficial embedding models for retrieval precision.

**Experiments**:
1. Embedding model precision: all-MiniLM-L6-v2 (384d, CPU) vs bge-large-en-v1.5 (1024d, GPU)
2. Retrieval quality vs cost tradeoff analysis

**Requirements**: CUDA GPU (any size), `sentence-transformers`, test corpus

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU detected — run the CPU notebook instead: notebook-exercise.ipynb\n"
        "To provision a GPU machine: see notes/06-ai-infrastructure/ch01-gpu-architecture/"
    )

device = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {device}  |  VRAM: {vram:.1f} GB")
print("✓ GPU available for accelerated embedding generation")

## Experiment — Embedding Model Precision Comparison

Compare retrieval quality between:
- **all-MiniLM-L6-v2**: 384 dimensions, 22M params, CPU-viable
- **bge-large-en-v1.5**: 1024 dimensions, 335M params, GPU-beneficial

**Metric**: Precision@5 on a test corpus with 10 held-out queries.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import time

# Test corpus (20 docs)
corpus = [
    "Transformers use self-attention to process sequences in parallel",
    "BERT is a bidirectional encoder from transformers",
    "GPT uses causal masking for autoregressive generation",
    "Retrieval-augmented generation grounds LLMs in external data",
    "Vector databases enable semantic search at scale",
    "Byte-pair encoding splits words into subword tokens",
    "Temperature controls the randomness of LLM sampling",
    "RLHF aligns models with human preferences",
    "Chain-of-thought prompting improves multi-step reasoning",
    "Embeddings map text to fixed-size dense vectors",
    "Convolutional neural networks excel at image classification",
    "Recurrent networks process sequences step by step",
    "Backpropagation computes gradients via the chain rule",
    "Dropout prevents overfitting by randomly zeroing activations",
    "Batch normalization stabilizes deep network training",
    "Adam optimizer adapts learning rates per parameter",
    "Cross-entropy loss is standard for classification",
    "Precision measures the accuracy of positive predictions",
    "F1 score balances precision and recall",
    "ROC curves plot true positive vs false positive rates"
]

# Test queries with relevant doc indices
test_queries = [
    ("What is self-attention?", [0]),
    ("Explain BERT architecture", [1]),
    ("How does RAG work?", [3]),
    ("What are vector databases for?", [4]),
    ("Explain tokenization", [5]),
    ("What is temperature in LLMs?", [6]),
    ("How does RLHF work?", [7]),
    ("What is chain-of-thought?", [8]),
    ("Define embeddings", [9]),
    ("What is backpropagation?", [12])
]

models = {
    'all-MiniLM-L6-v2': 384,
    'BAAI/bge-large-en-v1.5': 1024
}

results = {}

for model_name, dims in models.items():
    print(f"\n{'='*60}")
    print(f"Model: {model_name} ({dims}d)")
    print('='*60)

    # Load model on GPU
    model = SentenceTransformer(model_name, device='cuda')

    # Encode corpus
    start = time.time()
    corpus_embeddings = model.encode(corpus, convert_to_numpy=True)
    corpus_time = time.time() - start

    # Test queries
    precisions = []
    for query_text, relevant_indices in test_queries:
        query_emb = model.encode(query_text, convert_to_numpy=True)
        scores = cosine_similarity([query_emb], corpus_embeddings)[0]
        top5_indices = np.argsort(scores)[-5:][::-1]

        # Calculate precision@5
        hits = sum(1 for idx in top5_indices if idx in relevant_indices)
        precision = hits / 5.0
        precisions.append(precision)

    avg_precision = np.mean(precisions)
    avg_time = corpus_time / len(corpus)

    results[model_name] = {
        'dims': dims,
        'precision@5': avg_precision,
        'time_per_doc': avg_time
    }

    print(f"Precision@5: {avg_precision:.3f}")
    print(f"Avg encoding time: {avg_time*1000:.2f} ms/doc")

print("\n" + "="*60)
print("COMPARISON TABLE")
print("="*60)
print(f"{'Model':<30} {'Dims':>6} {'Precision@5':>12} {'ms/doc':>8}")
print("-"*60)
for model_name, stats in results.items():
    print(f"{model_name:<30} {stats['dims']:>6} {stats['precision@5']:>12.3f} {stats['time_per_doc']*1000:>8.2f}")

## Analysis

**Trade-offs**:
- **all-MiniLM-L6-v2**: Faster, smaller, good baseline precision, CPU-viable
- **bge-large-en-v1.5**: Higher precision, GPU-beneficial, 3-5× slower per doc

**When to choose**:
- Use MiniLM for prototyping, CPU deployments, or latency-critical applications
- Use bge-large when retrieval quality is paramount and GPU is available